#### Mengunduh request

In [1]:
!pip install requests beautifulsoup4 pandas lxml

#202431036_Fathur Rasyid Irimawa

#### Memanggil Library

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import csv
import re

#202431036_Fathur Rasyid Irimawa

#### Mengambil halaman web

In [2]:
url = "https://en.wikipedia.org/wiki/List_of_countries_by_external_debt"
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
response = requests.get(url, headers=headers)

print(f"Status Code: {response.status_code}")
if response.status_code == 200:
    print("Berhasil terhubung ke Wikipedia!\n")

#202431036_Fathur Rasyid Irimawa

Status Code: 200
Berhasil terhubung ke Wikipedia!



#### parser

In [3]:
soup = BeautifulSoup(response.text, 'html.parser')

#202431036_Fathur Rasyid Irimawa

#### Mencari data

In [5]:
tables = soup.find_all('table', class_='wikitable')
target_table = tables[0]
    
rows = target_table.find_all('tr')[1:]
print(f"Jumlah blok data (baris) ditemukan: {len(rows)}")
if len(rows) >= 200:
    print("Syarat minimal 200 data terpenuhi!\n")
    
#202431036_Fathur Rasyid Irimawa

Jumlah blok data (baris) ditemukan: 209
Syarat minimal 200 data terpenuhi!



#### Mengambil atribut dari setiap blok

In [6]:
result = []
for row in rows:
    cols = row.find_all(['td', 'th'])
            
    if len(cols) >= 5:
        country = cols[0].get_text(strip=True).split('[')[0]
        external_debt = cols[1].get_text(strip=True).split('[')[0]
        per_capita = cols[3].get_text(strip=True).split('[')[0]
        gdp_percentage = cols[4].get_text(strip=True).split('[')[0]
            
        result.append({
            'Country': country,
            'External_Debt': external_debt,
            'Per_Capita': per_capita,
            'GDP_Percentage': gdp_percentage
        })
#202431036_Fathur Rasyid Irimawa

#### Menampilkan 5 data pertama

In [7]:
print("Menampilkan 5 data pertama dari list hasil scraping:")
for i in range(5):
    print(result[i])
#202431036_Fathur Rasyid Irimawa

Menampilkan 5 data pertama dari list hasil scraping:
{'Country': 'United States', 'External_Debt': '40,598.133', 'Per_Capita': '93.25', 'GDP_Percentage': '21.59'}
{'Country': 'United Kingdom', 'External_Debt': '11,355', 'Per_Capita': '278.18', 'GDP_Percentage': '5.91'}
{'Country': 'France', 'External_Debt': '9,249.4815', 'Per_Capita': '257.21', 'GDP_Percentage': '57.85'}
{'Country': 'Germany', 'External_Debt': '8,139.6669', 'Per_Capita': '149.27', 'GDP_Percentage': '46.71'}
{'Country': 'Netherlands', 'External_Debt': '4,877.3109', 'Per_Capita': '336.43', 'GDP_Percentage': '100.17'}


#### Membuat file csv

In [8]:
csv_filename = "data_utang_negara.csv"
with open(csv_filename, mode='w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['Country', 'External_Debt', 'Per_Capita', 'GDP_Percentage'])
    writer.writeheader()
    for row_data in result:
        writer.writerow(row_data)
print(f"\nData mentah berhasil disimpan ke {csv_filename}")
#202431036_Fathur Rasyid Irimawa


Data mentah berhasil disimpan ke data_utang_negara.csv


#### Membaca file CSV dan menampilkan datanya

In [10]:
df = pd.read_csv(csv_filename)
print("\nData dibaca menggunakan Pandas (Tipe data awal masih berupa Object/Teks):")
print(df.head())
#202431036_Fathur Rasyid Irimawa


Data dibaca menggunakan Pandas (Tipe data awal masih berupa Object/Teks):
          Country External_Debt Per_Capita  GDP_Percentage
0   United States    40,598.133      93.25           21.59
1  United Kingdom        11,355     278.18            5.91
2          France    9,249.4815     257.21           57.85
3         Germany    8,139.6669     149.27           46.71
4     Netherlands    4,877.3109     336.43          100.17


#### Memastikan kolom angka bertipe int/float

In [11]:
def clean_to_numeric(text):
    if pd.isna(text):
        return None
    text = str(text).replace(',', '').strip()
    num = re.sub(r'[^\d.]', '', text)
    if num == '':
        return None
    return float(num)

df['External_Debt'] = df['External_Debt'].apply(clean_to_numeric)
df['Per_Capita'] = df['Per_Capita'].apply(clean_to_numeric)
df['GDP_Percentage'] = df['GDP_Percentage'].apply(clean_to_numeric)
    
df = df.dropna()

#202431036_Fathur Rasyid Irimawa

#### membuat kolom kategori logis berdasarkan persentase GDP

In [17]:
# Proses tambahan untuk soal nomor 4 decision tree
df['Risk_Category'] = df['GDP_Percentage'].apply(lambda x: 'Risiko Tinggi' if x > 60 else 'Risiko Rendah')
df.to_csv("data_utang_negara_clean.csv", index=False)
    
print("\nData setelah dibersihkan dan diubah menjadi tipe angka:")
print(df.dtypes)
print("\nTampilan Data Bersih Siap Pakai:")
print(df.head())

#202431036_Fathur Rasyid Irimawa


Data setelah dibersihkan dan diubah menjadi tipe angka:
Country            object
External_Debt     float64
Per_Capita        float64
GDP_Percentage    float64
Risk_Category      object
dtype: object

Tampilan Data Bersih Siap Pakai:
          Country  External_Debt  Per_Capita  GDP_Percentage  Risk_Category
0   United States     40598.1330       93.25           21.59  Risiko Rendah
1  United Kingdom     11355.0000      278.18            5.91  Risiko Rendah
2          France      9249.4815      257.21           57.85  Risiko Rendah
3         Germany      8139.6669      149.27           46.71  Risiko Rendah
4     Netherlands      4877.3109      336.43          100.17  Risiko Tinggi
